# Py Lab 5 — Data visualization I
**PUBHLT 0411 · Python for Public Health Data Analysis**

Name: 

Submit this notebook as a `.ipynb` file.

---

Each exercise states the task and names the variables to create. Write your code in the empty cell below it and run the cell.

Use the exact variable names given — later exercises build on them.

## Before you start — the data

This lab reads:

- `pa_asthma_prevalence.csv`
- `pa_counties.csv`
- `pa_covid_monthly.csv`
- `pa_maternal_infant.csv`

These live in a `data/` folder next to this notebook. Run the cell below once to confirm Python can see them before you begin.

In [ ]:
# Setup — run this cell once, before anything else.
#
# The lab's CSVs live in a `data/` folder next to this notebook. This cell
# does not download anything; it just checks Python can find them.
import os

FILES = [
    "pa_asthma_prevalence.csv",
    "pa_counties.csv",
    "pa_covid_monthly.csv",
    "pa_maternal_infant.csv"
]

missing = [f for f in FILES if not os.path.exists(os.path.join("data", f))]

if missing:
    print("MISSING from data/:", ", ".join(missing))
    print("Working directory:", os.getcwd())
    print("Save this notebook into python/, next to data/, reopen it, and re-run this cell.")
else:
    print("Data ready:", ", ".join(FILES))

---

## Exercise 1 — A histogram shows a distribution

Draw the distribution of **infant mortality** across the counties that publish a rate, and mark the **median** with a dashed line.

**Create:**

- `births`: `data/pa_maternal_infant.csv` read as a DataFrame (67 rows, one per county).
- `reporting`: the rows of `births` where `infant_mortality_per_1000` is not missing (44 rows; small counties have their rate suppressed).
- `state_median`: a number, the median of `infant_mortality_per_1000` across `reporting`.

**Draw:** a histogram of `infant_mortality_per_1000` for `reporting`, a dashed vertical line at `state_median`, and axis labels with units.

---

## Exercise 2 — Bars compare categories

Draw the five counties with the **lowest** adult smoking prevalence. Use `.nsmallest()`, and sort so the bars run in order.

**Create:**

- `asthma`: `data/pa_asthma_prevalence.csv` read as a DataFrame (67 rows).
- `bottom5`: the 5 rows of `asthma` with the lowest `smoking_prev_pct`, sorted lowest first, all columns kept.

**Draw:** one horizontal bar per county with `ax.barh()`, county names on the y-axis, the x-axis labelled with units, and a title.

---

## Exercise 3 — A boxplot compares groups

Draw a boxplot of adult **smoking** prevalence by **region**.

**Create:**

- `asthma`: as in Exercise 2.
- `counties`: `data/pa_counties.csv` read as a DataFrame (67 rows).
- `joined`: `asthma` left-merged on `county` with the `county` and `region` columns of `counties` (67 rows).
- `regions`: a list of the six region names in alphabetical order.
- `groups`: a list of six Series, one per region in the order of `regions`. Each holds that region's `smoking_prev_pct` values.

**Draw:** `ax.boxplot(groups)` with the boxes named by region, the title "Adult smoking by region", and a y-axis label with units at a font size above 10.

---

## Exercise 4 — Scatter shows co-movement

Draw infant mortality against **child poverty** for the reporting counties, and put the correlation in the title.

**Create:**

- `births`, `reporting`: as in Exercise 1.
- `r`: a number, the correlation between `children_in_poverty_pct` and `infant_mortality_per_1000` across `reporting`.

**Draw:** a scatter with child poverty on x and infant mortality on y, both labelled with units, and `r` to 3 decimals in the title.

---

## Exercise 5 — Mapping versus setting

With seaborn, draw **obesity** against **asthma**, and colour the points by **region**.

**Create:**

- `asthma`, `counties`, `joined`: as in Exercise 3.

**Draw:** `sns.scatterplot()` with `data=joined`, `obesity_prev_pct` on x, `asthma_prev_pct` on y, and `region` mapped to colour.

---

## Exercise 6 — Two series need two channels

Draw **asthma** against **smoking**, and show metro status with two channels at once (colour and marker shape), so the groups stay distinct even without colour.

**Create:**

- `asthma`, `counties`: as before.
- `joined`: `asthma` left-merged on `county` with the `county` and `metro` columns of `counties` (67 rows). This replaces Exercise 3's `joined`.

**Draw:** call `ax.scatter()` once per metro group, with `smoking_prev_pct` on x and `asthma_prev_pct` on y: Metro as circles (`"o"`) in `#003594`, Nonmetro as triangles (`"^"`) in `#8a6400`, each with `label=`. Then call `ax.legend()`, and label both axes with units.

---

## Exercise 7 — A line needs an ordered axis

Compare COVID-19 deaths in **metro and nonmetro** Pennsylvania, month by month, from August 2021 onward.

**Create:**

- `covid`: `data/pa_covid_monthly.csv`, with `month` converted to dates and only the rows from 2021-08-01 onward (2,144 rows).
- `counties`: as before.
- `merged`: `covid` merged on `county` with the `county`, `population_2023` and `metro` columns of `counties` (2,144 rows).
- `monthly`: one row per month and metro group (64 rows), with columns `month`, `metro`, `covid_deaths` (summed), `population_2023` (summed), and `per_100k` (deaths ÷ population × 100,000).
- `peaks`: for each metro group, the row of `monthly` with the highest `per_100k` (2 rows).

**Draw:** `sns.lineplot()` with `month` on x, `per_100k` on y, and `metro` mapped to both `hue=` and `style=`.

---

## Exercise 8 — Repair a truncated axis

**This figure runs without error and it is misleading.** A bar encodes value by length, so its baseline must be zero. `set_ylim(10.2, 10.9)` breaks the correspondence between length and value. Run the code below, then repair it in the empty cell.

**Create:**

- `asthma`, `counties`, `joined`: as in Exercise 3.
- `by_region`: a Series holding the mean `asthma_prev_pct` for each region, sorted ascending (6 values).
- `spread`: a number, `by_region`'s highest value minus its lowest.

**Draw:** the same figure, with `set_ylim(0, 12)` and a title that describes the chart in place of the `MISLEADING` one.

*The code this exercise refers to:*

In [ ]:
import pandas as pd                                        # tables
import matplotlib.pyplot as plt                            # plotting

asthma = pd.read_csv("data/pa_asthma_prevalence.csv")      # one row per county
counties = pd.read_csv("data/pa_counties.csv")             # region per county
joined = asthma.merge(counties[["county", "region"]],      # attach region ...
                      on="county", how="left")             # ... keeping every county

by_region = (joined.groupby("region")["asthma_prev_pct"]   # one mean per region ...
                   .mean().sort_values())                  # ... in ascending order

fig, ax = plt.subplots(figsize=(7, 3))                     # canvas and one axes
ax.bar(by_region.index, by_region.values, color="#003594") # one bar per region
ax.set_ylim(10.2, 10.9)                                    # <- the distortion
ax.set_ylabel("Mean adult asthma (%)")                     # y-axis label with units
ax.set_title("MISLEADING: regional asthma differences look large")  # title
ax.tick_params(axis="x", labelrotation=30, labelsize=7)    # tilt and shrink region names
plt.tight_layout()                                         # keep labels inside the canvas
plt.show()                                                 # render the figure